In [ ]:
import coflandscaper as cl

### Before you run this notebook (required)

- Place exactly one node XYZ in 0_node/ and one linker XYZ in 0_linker/.
- At the connection points, insert a dummy atom: **He** if node and linker are connected by a single bond and **Se** f node and linker are connected by a double bond .
- If your COF contains Se atoms, this workflow will not work with double bonds.
- This workflow is in general not tested for metal-containing COFs or MOFs.
- Choose a unique `COF_NAME`; all outputs will be saved in a folder named after the COF.

### Settings & options (what you can choose)

- `TOPOLOGY`: "hcb" or "sql".
- `BOND_TYPE`: "single" or "double" (must match your dummy atom choice).
- `COF_NAME`: "example-cof".
- `MODE`: choose what stacking modes to investigate: "incl" (inclined), "serr" (serrated), or "both".

In [2]:
TOPOLOGY = "hcb"
BOND_TYPE = "single" 
COF_NAME = "cof-1"
MODE = 'both'

### Optional settings

- `MACE_HEAD`: choose the MACE head for the mace-mh-1 model (default: "omat_pbe").
  - Model info: https://huggingface.co/mace-foundations/mace-mh-1
  - Recommended heads: "omat_pbe", "omol", "spice_wB97M", "matpes_r2scan".
- Set `MACE_HEAD` in the next cell to override the default.

In [ ]:
MACE_HEAD = "omat_pbe"

### Build single-layer COF + Pre-Optimization:

Creates the single-layer COF and then pre-optimizes it with MACE.

Defaults (advanced cells below):
- `MacePreopt`: `fmax=0.01`, `dtype="float64"`, `head="omat_pbe"`, `device="cpu"`, `fix_z=True`, `dispersion=False`.
- `input_folder`/`output_folder` default to `COF_NAME`/1_{`COF_NAME`}_single_layer (reads `{COF_NAME}_unopt.cif`, writes `{COF_NAME}_preopt.cif`).

Options (advanced cells below):
- `fmax`: force convergence threshold (lower = tighter, slower).
- `dtype`: numerical precision; use `"float64"` (more accurate, slower) or `"float32"` (faster, less accurate).
- `head`: MACE head to use (e.g., `"omat_pbe"`, `"omol"`, `"spice_wB97M"`, `"matpes_r2scan"`).
- `device`: compute device, e.g. `"cpu"` or `"cuda"` (if available).
- `fix_z`: if `True`, freezes atomic motion along $z$ so the layer stays flat. This can prevent twisting in a free single layer, which can otherwise lead to unphysical overlaps and unstable downstream structure generation. With `fix_z=True`, only in-plane positions relax while $z$ remains as in the initial pormake structure.


In [ ]:
# Defaults
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, bond_type=BOND_TYPE, cof_name=COF_NAME)

In [ ]:
# Configurable (all options)
builder = cl.BuildCOF2D()
builder.build(
    topo=TOPOLOGY,
    output_folder=None,
    bond_type=BOND_TYPE,
    cof_name=COF_NAME,
)

In [ ]:
# Defaults
preopt = cl.MacePreopt()
preopt.run(cof_name=COF_NAME)

In [ ]:
# Configurable (all options)
preopt = cl.MacePreopt(
    fmax=0.01,
    dtype="float64",
    head=MACE_HEAD,
    device="cpu",
    fix_z=True,
 )
preopt.run(
    cof_name=COF_NAME,
    input_folder=None,
    output_folder=None,
)

### Create ILD×ILS structure-matrix

Generate stacking variants by changing interlayer distance (ILD, along $z$) and interlayer slipping (ILS, in-plane).

Stacking Modes:
- Serrated: build a bilayer and shift the top layer back and forth around the AB shift.
- Inclined: tilt the $c$ vector to apply a continuous in-plane shift in one direction.

Defaults:
- Default max. slip length/angle correspond to AB stacking and are auto-computed (use `print_shift` to show them).
- Default ILD range is 3.0 to 4.5 in 0.1 A steps.
- Default ILS range is 0 to max in 1 A steps.
- Input defaults to `COF_NAME`/1_{`COF_NAME`}_single_layer/{`COF_NAME`}_preopt.cif.
- Output defaults to `COF_NAME`/2_{`COF_NAME`}_matrix/{serr|incl}.

Outputs go to `COF_NAME`/2_{`COF_NAME`}_matrix/serr or /incl depending on `MODE`. These structures are used for single-point energy evaluation and landscape plots.

Advanced options (configurable cell):
- ILD range: `ild_start`, `ild_end`, `ild_step`.
- ILS range: `ils_length_start`, `ils_length_end`, `ils_length_step`.
- Direction: `ils_angle` (degrees).
- Paths: `input_cif`, `output_base`.
- `print_shift` to display the auto-computed AB shift.

In [ ]:
# Defaults
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

In [ ]:
# Configurable (all options)
matrix = cl.CreateMatrix(
    ild_start=3.0,
    ild_end=4.5,
    ild_step=0.1,
    ils_length_start=0.0,
    ils_length_end=None,
    ils_length_step=1.0,
    ils_angle=None,
    print_shift=True,
 )
matrix.run(
    cof_name=COF_NAME,
    topo=TOPOLOGY,
    mode=MODE,
    input_cif=None,
    output_base=None,
)

### Optional: MACE pre-screening (can be skipped)

You can skip the MACE step entirely and go straight to DFT.  
However, running MACE first is a cheap way to get a **rough map of the energy landscape** (relative stability trends, likely low-energy motifs, obviously high-energy/unstable regions).

Use it as a **pre-screen** to:
- discard clearly unfavorable regions of configuration space,
- focus DFT on the most promising/interesting candidates,

which can **significantly reduce the number (and cost) of subsequent DFT calculations**.

### MACE single-point energies

Compute locally on this machine single-point energies for each structure to build the energy landscape. No geometry optimization is performed.

Defaults:
- Uses the MACE head set in `MACE_HEAD` ("omat_pbe" by default).
- Dispersion defaults by head (`omat_pbe`/`matpes_r2scan` = on; `omol`/`spice` = off).
- Results are saved in `COF_NAME`/3_{`COF_NAME`}_landscape as {`COF_NAME`}_sp_energies_{serr|incl}.csv.

Options:
- `dtype`: numerical precision; use `"float64"` (more accurate, slower) or `"float32"` (faster, less accurate).
- `head`: MACE head to use (e.g., `"omat_pbe"`, `"omol"`, `"spice_wB97M"`, `"matpes_r2scan"`).
- `device`: compute device, e.g. `"cpu"` or `"cuda"` (if available).
- `dispersion`: manually override dispersion (True/False).
- `input_folder`, `output_csv_dir` in `run_mode` to override default paths.

In [ ]:
# Defaults
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
sp = cl.MaceSP(
    device="cpu",
    dtype="float64",
    head=MACE_HEAD,
    dispersion=None,
)
sp.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=None,
    output_csv_dir=None,
)

### DFT single-point inputs (Crystal23)

This section generates CRYSTAL .d12 input files for each structure.
You must run these .d12 files externally on your HPC.
After the runs finish, copy the resulting .out files back into the same folders as the .d12 files (each structure has its own folder).

Default Level of Theory: HSEsol-3c/sol-mSVP

In [ ]:
# Defaults
sp = cl.CrystalSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
sp = cl.CrystalSP(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    post_block=None,
 )
sp.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=None,
    output_base=None,
)

### Read CRYSTAL energies

Run this after the CRYSTAL jobs finish and the .out files are placed next to their .d12 inputs.
This will extract energies and write a CSV in 3_{COF_NAME}_landscape.

In [ ]:
# Defaults
sp = cl.CrystalSP()
sp.read_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
sp = cl.CrystalSP()
sp.read_mode(
    cof_name=COF_NAME,
    mode=MODE,
    output_csv_dir=None,
    input_base=None,
)

### Potential Energy Landscape (PES)

The PES is an approximation that treats each structure as identical except for its `ILD`/`ILS` values. Under this assumption, the landscape provides a reasonable qualitative map of relative energies across stacking configurations. The goal is to identify candidate minima that can then be validated and refined with full geometry optimization (MACE or DFT).

Options (`Landscape.run_mode`):
- `input_folder`: optional explicit folder for one mode; default uses `COF_NAME`/2_{`COF_NAME`}_matrix/{`serr`|`incl`} from `mode`.
- `output_folder`: where plots are written; defaults to `COF_NAME`/3_{`COF_NAME`}_landscape.
- `colorscheme`: one of `grey`, `colorblind` (uses `cividis`), or `viridis` (default).
- `plot_mode`: `heatmap`, `isolines`, or `both`.
- `rel_energy_max`: cap relative energies in eV; values above are clipped in plots.
- `show_minima_markers`: if `True` (default), show red/green minima markers on plots.

Defaults:
- `colorscheme` defaults to `viridis`.
- `plot_mode` defaults to `both`.

In [ ]:
# Defaults
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
landscape = cl.Landscape()
landscape.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=None,
    output_folder=None,
    colorscheme="viridis",
    plot_mode="both",
    rel_energy_max=None,
    show_minima_markers=True,
 )

Add extra (ILD, ILS) points you want to include in the selection. Use a list of tuples in Angstrom: [(ILD, ILS), ...]. These are added on top of the automatically detected minima for each mode (serr/incl).

In [ ]:
EXTRA_SERR = [(3.6, 0.0)]
EXTRA_INCL = [(3.6, 0.0)]

Selection copies CIFs at the auto-detected minima (and any extra ILD/ILS pairs you provide) into the selection folders for downstream optimization.

In [ ]:
# Defaults
selector = cl.SelectCofs()
selector.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    selections_serr=EXTRA_SERR,
    selections_incl=EXTRA_INCL,
 )

In [ ]:
# Configurable (all options)
selector = cl.SelectCofs()
selector.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    selections_serr=EXTRA_SERR,
    selections_incl=EXTRA_INCL,
    include_autoselect=True,
    input_base=None,
    output_base=None,
 )

### DFT geometry optimization inputs (Crystal23)

This section generates CRYSTAL .d12 inputs for geometry optimization using the selected structures.
Inputs are taken from `COF_NAME`/3_{`COF_NAME`}_landscape/selection/{serr|incl}.
Outputs are written to `COF_NAME`/4_{`COF_NAME`}_final_structures/dft_{serr|incl}.

You must run these .d12 files externally on your HPC.
After the runs finish, copy the resulting .out files back into the same folders as the .d12 files (each structure has its own folder).

In [ ]:
# Defaults
opt = cl.CrystalOpt()
opt.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
opt = cl.CrystalOpt(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    maxtradius="0.8",
    post_block=None,
 )
input_base = None
output_base = None
opt.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=input_base,
    output_base=output_base,
)

### Extract Crystal23 optimized structures

Run this after the jobs finish and the .out files are placed next to their .d12 inputs in `COF_NAME`/4_{`COF_NAME`}_final_structures/dft_{serr|incl}.
This extracts final CIF structures from CRYSTAL outputs into the corresponding dft_ mode folders.

In [ ]:
# Defaults
opt = cl.CrystalOpt()
opt.extract_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
opt = cl.CrystalOpt()
opt.extract_mode(
    cof_name=COF_NAME,
    mode=MODE,
    output_base=None,
    input_base=None,
)

### Analyze + Visualize

`analyze` computes ILD/ILS for the final optimized structures and writes `final_structures.csv` in the output folder (defaults to `COF_NAME`/4_{`COF_NAME`}_final_structures). Use `input_base` and `output_base` to override input/output locations, and `print_values` to control stdout logging.

`visualizecof` opens py3Dmol views for the final structures and prints ILD/ILS in the console. Inputs default to `COF_NAME`/4_{`COF_NAME`}_final_structures, with per-mode supercell sizes controlled by `supercell_size_serr` and `supercell_size_incl`. Viewer options include `width`, `height`, `background`, `style`, and `add_unit_cell`.

In [ ]:
# Defaults
analyze = cl.analyze
analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
analyze = cl.analyze
analyze(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=None,
    output_base=None,
    print_values=True,
 )

In [ ]:
# Defaults
visualize = cl.visualizecof
visualize(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
visualize = cl.visualizecof
visualize(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=None,
    width=800,
    height=600,
    background="white",
    style="stick",
    add_unit_cell=True,
    supercell_size_serr=(2, 2, 1),
    supercell_size_incl=(2, 2, 2),
)